# Qwen2.5 and SmolLM2: base versus instruct
Four checkpoints, identical plain-text association prompts, original OLMo predictors unchanged.
**Run All only creates the controls. Click Launch/resume to start.** Stop and New run never execute automatically.
Use the existing OLMo/Pythia environment. This is a separate experiment; do not change the running studies.


In [ ]:
from pathlib import Path
import json
import paired_suite as study

OLMO_SOURCE = Path('/home/ubuntu/1/runs/olmo_association_v1')
PREVIOUS = Path('/home/ubuntu/4/runs/multimodel_paper_v1')
# PREVIOUS can instead be your uploaded multimodel_share.zip.
OUTPUT = Path.cwd() / 'runs' / 'paired_base_instruct_v1'
REMEMBER = Path.cwd() / '.paired_base_instruct_last.json'

if REMEMBER.exists():
    OUTPUT = Path(json.loads(REMEMBER.read_text())['output'])
    SETTINGS = json.loads((OUTPUT / 'settings.json').read_text())
else:
    SETTINGS = study.defaults(OLMO_SOURCE, PREVIOUS, OUTPUT)

# Full trajectories for three seeds; detailed paths for seed 1 only by default.
# To measure paths on every seed, set this BEFORE the first launch:
# SETTINGS['path_seeds'] = [1, 2, 3]
# SETTINGS['hours'] = 11.5

OTHER_GPU_RUNS = [
    Path('/home/ubuntu/4/runs/frame_history_v1'),
    Path('/home/ubuntu/4/runs/multimodel_paper_v1'),
]


In [ ]:
def remember_output(path):
    global OUTPUT, SETTINGS
    OUTPUT = Path(path)
    SETTINGS = json.loads((OUTPUT / 'settings.json').read_text())
    REMEMBER.write_text(json.dumps({'output': str(OUTPUT)}))

def check_other_workers():
    active = [str(p) for p in OTHER_GPU_RUNS
              if p.resolve() != OUTPUT.resolve() and study.f.ng.alive(p)]
    if active:
        raise RuntimeError('Another GPU study is active. Stop it in its own notebook first: ' + ', '.join(active))

def launch_run():
    check_other_workers()
    remember_output(study.launch(SETTINGS))

def fresh_run():
    check_other_workers()
    remember_output(study.restart(SETTINGS))

def refresh_run():
    if OUTPUT.exists():
        _ = study.refresh(OUTPUT)
    else:
        print('Not started. Click Launch/resume.')

def show_results():
    study.show(OUTPUT)

def stop_run():
    if OUTPUT.exists():
        study.stop(OUTPUT)
    else:
        print('No run to stop.')

def export_results():
    if not (OUTPUT / 'settings.json').exists():
        print('No run to export yet.')
        return
    print(study.export(OUTPUT))


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    panel = widgets.Output()
    buttons = []
    for label, action in [
        ('Launch/resume', launch_run), ('Refresh', refresh_run),
        ('Show results', show_results), ('Stop', stop_run),
        ('Export', export_results), ('New run', fresh_run),
    ]:
        button = widgets.Button(description=label)
        def clicked(_, action=action):
            with panel:
                clear_output(wait=True)
                try:
                    action()
                except Exception as exc:
                    print(type(exc).__name__ + ': ' + str(exc))
        button.on_click(clicked)
        buttons.append(button)
    display(widgets.VBox([widgets.HBox(buttons[:3]), widgets.HBox(buttons[3:]), panel]))
except ImportError:
    print('Widgets unavailable. Run launch_run() in a new cell to start; refresh_run() for status.')
    print('Other manual actions: show_results(), stop_run(), export_results(), fresh_run().')


## What to inspect
- `paired_comparison.json`: acquisition quality and retention for each base/instruct pair.
- `transfer_metrics.json`: frozen OLMo predictors, both current-update diagnostics and one-update-ahead prediction.
- `REPORT.txt` and plots: concise results.
- `runs/.../seed.../path_accounting.json` is nested under the selected event directories: inspect quadrature and slope audits. Completion is not numerical convergence.
- `paired_models_share.zip`: fresh snapshot created automatically at pause/completion or with Export.

No native-chat-format condition or matched-acquisition retuning is included. This is the fixed-budget, matched-plain-text comparison. The full queue may take multiple sessions.
